# E-Tobit 模型的 GMM 估计

> 对应代码：`MathStatsCode/code_in_notes/Chap.10/GMM_ettobit.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：构造指数变换的 Tobit 型数据，用 x、x²、x³、eˣ、sin x 等多个矩条件进行 GMM 估计。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear`、`set more off`，并以 `set obs 500`、`set seed 8997` 准备500个观测与固定种子。


In [ ]:
%%stata
clear
set more off
set obs 500
set seed 8997


**生成数据**

生成**E-Tobit 型数据**：设定真值 α=5、β=3、σ=2（用 `local` 宏存放）；`gen y = exp(α+βx+ε)-a` 是核心公式——先由对数正态形式生成潜变量，再减去未知常数 a=10。由于存在“平移”参数 a，直接看 y 无法线性化，这正是需要 GMM 的原因。


In [ ]:
%%stata
gen x=rnormal(3,2)
local a=10
local alpha=5
local beta=3
local sigma=2
gen y = exp(`alpha'+`beta'*x+rnormal()*sqrt(`sigma'))-`a'


**生成工具**

构造**工具变量**：`x2=x²`、`x3=x³`、`x_e=eˣ`、`x_sin=sin(x)`——它们都与 x 相关、与误差项 ε 独立，可以充当矩条件中的工具变量。工具变量的个数多于参数个数（过度识别），这是后面检验的基础。


In [ ]:
%%stata
gen x2=x^2
gen x3=x^3
gen x_e=exp(x)
gen x_sin=sin(x)


**计算a的初始值**

为参数 a 寻找一个合理的**初始值**：模型要求 `log(a+y)` 对所有观测都有定义，因此必须 a+y>0，即 a>-min(y)。程序取 `a_init=-min(y)+5`，留出一定余量，避免优化算法在迭代初期遇到对数为负而无法计算。`su y` 先取得 `r(min)`。


In [ ]:
%%stata
su y
local a_init=-1*r(min)+5


**GMM估计**

用 `gmm` 命令实现估计，共设6个矩条件：第1个 `log({a}+y)-{alpha}-{beta}*x` 来自模型的结构方程（其总体均值应为0）；后5个是同一结构误差分别与 x、x²、x³、eˣ、sin(x) 正交的矩条件。参数写在 `{}` 中并以初始值赋值（`{a=\`a_init'}` 等）。5个“过度识别”约束可用于检验模型设定是否正确（`estat overid`）；`winit(identity)` 指定第一步使用单位权重矩阵。


In [ ]:
%%stata
gmm (log({a=`a_init'}+y)-{alpha=0}-{beta=0}*x) ((log({a}+y)-{alpha}-{beta}*x)*x) ((log({a}+y)-{alpha}-{beta}*x)*x2) ((log({a}+y)-{alpha}-{beta}*x)*x3) ((log({a}+y)-{alpha}-{beta}*x)*x_e) ((log({a}+y)-{alpha}-{beta}*x)*x_sin), winit(identity)
